# Stage 1b — Local smoke tests: the GREEN SIGNAL gate

Report reference: `PROJECT_REPORT.md` §R4, §R5. Orchestration only (decision D13) — this notebook runs `scripts/smoke_test.py`, which in turn runs `pytest` over `tests/unit`, `tests/property`, `tests/theory`, `tests/smoke`, `tests/isolation`.

**The GREEN SIGNAL is a technical readiness gate, not evidence.** It says the implementation is correct enough that a training run will measure what it is intended to measure — it says nothing about whether the attack works, whether RCE helps, or whether the paper's hypothesis is true.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"Repo root: {REPO_ROOT}")

## Run the tiny end-to-end training smoke test (S15) directly, for visibility

The full gate is run by the next cell via `scripts/smoke_test.py`; this cell exercises `safelie.experiment.run_experiment_with_oracle` once, inline, so its artifact output is visible in this notebook rather than only inside pytest's capture.

In [ ]:
from safelie.experiment import run_experiment_with_oracle
from safelie.utils.config import load_experiment_config

cfg = load_experiment_config(str(REPO_ROOT / "configs" / "experiment" / "smoke.yaml"))
out_dir = run_experiment_with_oracle(cfg)
print(f"Tiny end-to-end run complete. Artifacts: {out_dir}")
print(f"  rounds.jsonl exists: {(out_dir / 'rounds.jsonl').exists()}")
print(f"  oracle.jsonl exists: {(out_dir / 'oracle.jsonl').exists()}")

## Run the full GREEN SIGNAL gate

This runs every CRITICAL local suite (S1-S28) and reports the eight readiness conditions G-1..G-8.

In [ ]:
result = subprocess.run(
    [sys.executable, str(REPO_ROOT / "scripts" / "smoke_test.py")],
    cwd=REPO_ROOT,
)
print(f"\nsmoke_test.py exit code: {result.returncode} ({'GREEN' if result.returncode == 0 else 'RED'})")

## What happens next

If GREEN: proceed to `colab_full_experiment.ipynb` on a Colab T4 High-RAM runtime for the Stage-2 compact pilot (`PROJECT_REPORT.md` §R7-§R8). If RED: fix the failing test(s), re-run the *whole* suite, and re-evaluate — a partially passing suite is a red signal (§R5.3).